# 📊 Data Dashboards in Python

Building professional interactive dashboards with Matplotlib, Plotly, and Altair.

**Sections:**
1. Matplotlib multi-panel dashboard
2. Plotly Express — interactive charts
3. Plotly Subplots — multi-metric dashboard
4. Altair — declarative charts
5. Sales performance dashboard (end-to-end)
6. Time-series dashboard
7. KPI summary cards with Plotly
8. Animated charts

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker

# Plotly: interactive charts that work in Jupyter
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Suppress matplotlib warnings
import warnings; warnings.filterwarnings('ignore')

# Set consistent style
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 10})

np.random.seed(42)
print('Libraries ready.')

## 1. Generating Realistic Sample Data

In [ ]:
# ── Generate Realistic Business Data ─────────────────────────────────────────

# 2 years of daily data
dates = pd.date_range('2023-01-01', periods=730, freq='D')
n = len(dates)

# Revenue: upward trend + weekly seasonality + noise
trend     = np.linspace(10_000, 18_000, n)
weekly    = 2000 * np.sin(2 * np.pi * np.arange(n) / 7)   # weekly cycle
noise     = np.random.normal(0, 600, n)
revenue   = trend + weekly + noise

# Customers: correlated with revenue
customers = (revenue / 50 + np.random.normal(0, 5, n)).astype(int).clip(0)

# Costs: 60% of revenue + fixed overhead
costs = revenue * 0.6 + 2000 + np.random.normal(0, 200, n)
profit = revenue - costs

# NPS score: rolling 30-day average, improving over time
nps_daily = np.random.normal(42 + np.linspace(0, 15, n), 8)
nps_daily = np.clip(nps_daily, -100, 100)

df = pd.DataFrame({
    'date': dates,
    'revenue':   revenue,
    'costs':     costs,
    'profit':    profit,
    'customers': customers,
    'nps':       nps_daily,
})

# Monthly aggregation for bar charts
df_monthly = df.resample('ME', on='date').agg({
    'revenue':   'sum',
    'costs':     'sum',
    'profit':    'sum',
    'customers': 'sum',
    'nps':       'mean',
})

print(df.head(3).to_string())
print(f'\nShape: {df.shape}')

## 2. Matplotlib Multi-Panel Executive Dashboard

In [ ]:
# ── Matplotlib Multi-Panel Dashboard ─────────────────────────────────────────

# Create a 2×3 grid with unequal sizes using GridSpec
fig = plt.figure(figsize=(16, 9))
fig.patch.set_facecolor('#0d1117')   # dark background
gs  = gridspec.GridSpec(2, 3, hspace=0.45, wspace=0.35)

# Colour palette
C = {'green': '#34d399', 'blue': '#60a5fa', 'orange': '#f59e0b',
     'pink': '#f472b6', 'purple': '#a78bfa', 'bg': '#161b22', 'text': '#e2e8f0'}

def dark_ax(ax):
    """Apply dark theme to a matplotlib axis."""
    ax.set_facecolor(C['bg'])
    ax.tick_params(colors=C['text'])
    ax.xaxis.label.set_color(C['text'])
    ax.yaxis.label.set_color(C['text'])
    ax.title.set_color(C['text'])
    for spine in ax.spines.values():
        spine.set_edgecolor('#30363d')
    ax.grid(True, color='#21262d', linestyle='--', alpha=0.6)
    return ax

# --- Panel 1 (top-left wide): Revenue trend with 30-day rolling average ---
ax1 = dark_ax(fig.add_subplot(gs[0, :2]))
ax1.plot(df['date'], df['revenue'], alpha=0.25, color=C['blue'], lw=0.8, label='Daily')
ax1.plot(df['date'], df['revenue'].rolling(30).mean(), color=C['blue'], lw=2, label='30-day avg')
ax1.fill_between(df['date'], df['revenue'].rolling(30).mean(), alpha=0.1, color=C['blue'])
ax1.set_title('Daily Revenue'); ax1.set_ylabel('$')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
ax1.legend(facecolor=C['bg'], labelcolor=C['text'])

# --- Panel 2 (top-right): Monthly profit bar chart ---
ax2 = dark_ax(fig.add_subplot(gs[0, 2]))
colors_bar = [C['green'] if p > 0 else '#ef4444' for p in df_monthly['profit']]
ax2.bar(range(len(df_monthly)), df_monthly['profit']/1000, color=colors_bar, alpha=0.85)
ax2.set_title('Monthly Profit'); ax2.set_ylabel('$K')

# --- Panel 3 (bottom-left): Revenue vs Costs stacked area ---
ax3 = dark_ax(fig.add_subplot(gs[1, 0]))
ax3.fill_between(df['date'], df['revenue'].rolling(14).mean(),
                 df['costs'].rolling(14).mean(), alpha=0.3, color=C['green'], label='Profit')
ax3.plot(df['date'], df['revenue'].rolling(14).mean(), color=C['green'], lw=1.5, label='Revenue')
ax3.plot(df['date'], df['costs'].rolling(14).mean(), color=C['orange'], lw=1.5, linestyle='--', label='Costs')
ax3.set_title('Revenue vs Costs (14d avg)')
ax3.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
ax3.legend(facecolor=C['bg'], labelcolor=C['text'], fontsize=8)

# --- Panel 4 (bottom-middle): Customer volume histogram ---
ax4 = dark_ax(fig.add_subplot(gs[1, 1]))
ax4.hist(df['customers'], bins=40, color=C['purple'], alpha=0.8, edgecolor='none')
ax4.axvline(df['customers'].mean(), color='white', lw=1.5, linestyle='--', label=f'Mean={df["customers"].mean():.0f}')
ax4.set_title('Customer Distribution'); ax4.set_xlabel('Daily Customers')
ax4.legend(facecolor=C['bg'], labelcolor=C['text'], fontsize=8)

# --- Panel 5 (bottom-right): NPS over time ---
ax5 = dark_ax(fig.add_subplot(gs[1, 2]))
nps_roll = df['nps'].rolling(30).mean()
ax5.plot(df['date'], nps_roll, color=C['pink'], lw=2)
ax5.axhline(0, color='#475569', lw=0.8, linestyle='--')
ax5.fill_between(df['date'], nps_roll, 0, where=(nps_roll > 0), alpha=0.15, color=C['green'])
ax5.fill_between(df['date'], nps_roll, 0, where=(nps_roll < 0), alpha=0.15, color='#ef4444')
ax5.set_title('NPS (30-day rolling)')

fig.suptitle('Executive Dashboard — 2 Year Overview', fontsize=14,
             color=C['text'], fontweight='bold', y=0.98)
plt.show()

## 3. Plotly Interactive Sales Dashboard

In [ ]:
# ── Plotly Interactive Multi-Metric Dashboard ─────────────────────────────────

# Create 2×2 subplot grid
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Monthly Revenue & Costs', 'Daily Profit (30d avg)',
                    'Revenue by Month (Waterfall)', 'NPS Trend'),
    vertical_spacing=0.15,
    horizontal_spacing=0.1,
)

# Panel 1: Grouped bar — Revenue vs Costs
months = df_monthly.index.strftime('%b %Y')
fig.add_trace(go.Bar(x=months, y=df_monthly['revenue']/1000,
                     name='Revenue', marker_color='#34d399', opacity=0.85), row=1, col=1)
fig.add_trace(go.Bar(x=months, y=df_monthly['costs']/1000,
                     name='Costs', marker_color='#f59e0b', opacity=0.85), row=1, col=1)

# Panel 2: Line — Daily profit smoothed
profit_smooth = df['profit'].rolling(30).mean()
fig.add_trace(go.Scatter(x=df['date'], y=profit_smooth,
                         fill='tozeroy', fillcolor='rgba(52,211,153,0.12)',
                         line=dict(color='#34d399', width=2),
                         name='Profit'), row=1, col=2)

# Panel 3: Waterfall chart — MoM revenue change
mom_delta = df_monthly['revenue'].diff()
fig.add_trace(go.Waterfall(
    x=months, y=mom_delta/1000,
    increasing=dict(marker_color='#34d399'),
    decreasing=dict(marker_color='#ef4444'),
    name='MoM Δ Revenue ($K)'), row=2, col=1)

# Panel 4: NPS scatter + smoothed line
nps_smooth = df['nps'].rolling(30).mean()
fig.add_trace(go.Scatter(x=df['date'], y=df['nps'], mode='markers',
                         marker=dict(color='#a78bfa', size=2, opacity=0.3),
                         name='NPS daily'), row=2, col=2)
fig.add_trace(go.Scatter(x=df['date'], y=nps_smooth,
                         line=dict(color='#a78bfa', width=2.5),
                         name='NPS 30d avg'), row=2, col=2)

# Apply dark theme
fig.update_layout(
    template='plotly_dark',
    title=dict(text='Interactive Business Dashboard', font=dict(size=18)),
    height=700, showlegend=True,
    paper_bgcolor='#0d1117', plot_bgcolor='#161b22',
    barmode='group',
)
fig.show()

## 4. KPI Cards with Plotly Indicator

In [ ]:
# ── KPI Summary Cards ─────────────────────────────────────────────────────────
# Plotly's Indicator trace creates beautiful KPI cards with delta arrows

# Compute KPIs: current quarter vs previous quarter
q_now  = df[df['date'] >= '2024-07-01']
q_prev = df[(df['date'] >= '2024-04-01') & (df['date'] < '2024-07-01')]

kpis = [
    ('Total Revenue',  q_now['revenue'].sum(),    q_prev['revenue'].sum(),    '$', True),
    ('Total Profit',   q_now['profit'].sum(),     q_prev['profit'].sum(),     '$', True),
    ('Total Customers',q_now['customers'].sum(),  q_prev['customers'].sum(),  '',  True),
    ('Avg NPS',        q_now['nps'].mean(),       q_prev['nps'].mean(),       '',  True),
]

fig = make_subplots(
    rows=1, cols=4,
    specs=[[{'type':'indicator'}]*4]
)

for i, (title, val, ref, prefix, higher_is_better) in enumerate(kpis, 1):
    fmt = f'{prefix},.0f' if prefix == '$' else ',.0f'
    fig.add_trace(go.Indicator(
        mode='number+delta+gauge',
        value=val,
        delta=dict(reference=ref, increasing_color='#34d399',
                   decreasing_color='#ef4444', valueformat=fmt),
        title=dict(text=title, font=dict(size=13)),
        number=dict(valueformat=fmt, font=dict(size=24)),
        gauge=dict(axis=dict(visible=False),
                   bar=dict(color='#34d399' if val >= ref else '#ef4444')),
    ), row=1, col=i)

fig.update_layout(
    template='plotly_dark', height=250,
    paper_bgcolor='#0d1117',
    title=dict(text='Q3 2024 KPIs vs Q2 2024', font=dict(size=16)),
)
fig.show()

## 5. Animated Bar Chart Race

In [ ]:
# ── Animated Bar Chart Race with Plotly ──────────────────────────────────────
# Show monthly sales by product category as an animated racing bar chart

# Generate per-product monthly revenue
products = ['Product A', 'Product B', 'Product C', 'Product D', 'Product E']
months_list = pd.date_range('2023-01-01', periods=24, freq='ME')

records = []
base = {'Product A': 5000, 'Product B': 3500, 'Product C': 4200,
        'Product D': 2800, 'Product E': 3100}
cumulative = {p: 0 for p in products}

for i, month in enumerate(months_list):
    for prod in products:
        # Simulate growing sales with different growth rates
        growth = 1 + 0.03 * i + np.random.normal(0, 0.05)
        monthly_rev = base[prod] * growth + np.random.normal(0, 300)
        cumulative[prod] += monthly_rev
        records.append({'Month': month.strftime('%b %Y'),
                        'Product': prod,
                        'Revenue': cumulative[prod]})

df_race = pd.DataFrame(records)

# Animated bar chart using Plotly Express
fig = px.bar(
    df_race,
    x='Revenue', y='Product', color='Product',
    animation_frame='Month',
    orientation='h',
    template='plotly_dark',
    title='Cumulative Revenue Race — By Product',
    color_discrete_sequence=['#34d399','#60a5fa','#a78bfa','#f59e0b','#f472b6'],
    text='Revenue',
)
fig.update_traces(texttemplate='$%{text:,.0f}', textposition='outside')
fig.update_layout(height=380, paper_bgcolor='#0d1117', plot_bgcolor='#161b22',
                  showlegend=False)
fig.show()